# BART Evaluation (ROUGE + BERTScore)

Official evaluation run for the milestone. Generates predictions from the
fine-tuned BART + LoRA checkpoint (from `01_train_bart.ipynb`) on a held-out
**test** subset -- deliberately not the validation set, since validation was
already monitored during training (see the loss curve in `01_train_bart.ipynb`).
Then runs `evaluation/evaluate.py reference` mode (ROUGE-1/2/L + BERTScore
against the real `Summary` column), per `docs/methodology.md`.

Self-contained: mounts Drive, clones/pulls the repo, installs dependencies,
and restores both the processed data and the trained checkpoint from Drive
backup if not already present locally.

In [1]:
# 1. Full environment setup: mount Drive, clone/pull repo, install deps, load keys,
# confirm GPU, and restore processed data + trained checkpoint from Drive backup
# if not already present locally.
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_DIR = '/content/Review-Summarization-LLM'
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    %cd /content
    !git clone https://github.com/jrsheffie/Review-Summarization-LLM.git
    %cd {REPO_DIR}

!pip install -q -r requirements.txt
!pip install -q -U torchao

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
github_token = userdata.get('GITHUB_TOKEN')
!git config --global user.email "jrsheffie@gmail.com"
!git config --global user.name "jrsheffie"

import torch
print('CUDA available:', torch.cuda.is_available())

BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'

if not os.path.exists('data/processed/test.csv'):
    print('test.csv not found locally -- restoring from Drive backup...')
    !mkdir -p data/processed data/raw
    !cp {BACKUP_DIR}/clean_reviews.csv {BACKUP_DIR}/train.csv {BACKUP_DIR}/val.csv {BACKUP_DIR}/test.csv {BACKUP_DIR}/product_batches.json data/processed/ 2>/dev/null
    !cp {BACKUP_DIR}/Reviews.csv data/raw/ 2>/dev/null

if not os.path.exists('outputs/bart_lora_30k/final'):
    print('Trained checkpoint not found locally -- restoring from Drive backup...')
    !mkdir -p outputs/bart_lora_30k/final
    !cp -r {BACKUP_DIR}/bart_lora_30k/* outputs/bart_lora_30k/final/

assert os.path.exists('data/processed/test.csv'), "test.csv still missing -- check Drive backup path/contents."
assert os.path.exists('outputs/bart_lora_30k/final'), "Checkpoint still missing -- check Drive backup path/contents."
print('Environment ready.')

Mounted at /content/drive
/content
Cloning into 'Review-Summarization-LLM'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 169 (delta 77), reused 93 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 115.28 KiB | 6.78 MiB/s, done.
Resolving deltas: 100% (77/77), done.
/content/Review-Summarization-LLM
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 22.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 998.3/998.3 kB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.9 MB/s eta 0:00:00
CUDA available: True
test.csv not found locally -- restoring from Drive backup...
Trained checkpoint not found locally -- restoring from Drive backup...
Environment ready.

In [2]:
# 2. Create a fixed, reproducible test subset (held out -- never seen during training or val monitoring)
import pandas as pd

TEST_SUBSET_SIZE = 3000
SEED = 42

test_df = pd.read_csv('data/processed/test.csv', engine='python', on_bad_lines='warn')
test_df = test_df.dropna(subset=['Text', 'Summary'])
test_subset = test_df.sample(n=min(TEST_SUBSET_SIZE, len(test_df)), random_state=SEED)

test_subset.to_csv('data/processed/test_subset_30k.csv', index=False)
print(f'Test subset: {len(test_subset)} rows')

Test subset: 3000 rows


In [3]:
# 3. Load the fine-tuned checkpoint and generate predictions on the test subset
import torch
from transformers import BartTokenizer, BartForConditionalGeneration
from peft import PeftModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

base_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')
tokenizer = BartTokenizer.from_pretrained('outputs/bart_lora_30k/final')
model = PeftModel.from_pretrained(base_model, 'outputs/bart_lora_30k/final')
model.to(device)
model.eval()

predictions = []
batch_size = 16

texts = test_subset['Text'].astype(str).tolist()

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    inputs = tokenizer(batch, return_tensors='pt', max_length=1024, truncation=True, padding=True).to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=64, num_beams=4, early_stopping=True)
    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    predictions.extend(decoded)
    if (i + batch_size) % 320 == 0 or (i + batch_size) >= len(texts):
        print(f'{min(i + batch_size, len(texts))}/{len(texts)}')

test_subset = test_subset.copy()
test_subset['prediction'] = predictions

import os
os.makedirs('outputs/summaries', exist_ok=True)
test_subset[['prediction', 'Summary']].to_csv('outputs/summaries/bart_predictions.csv', index=False)
print('Saved predictions for', len(test_subset), 'examples')

Using device: cuda


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

320/3000
640/3000
960/3000
1280/3000
1600/3000
1920/3000
2240/3000
2560/3000
2880/3000
3000/3000
Saved predictions for 3000 examples


In [4]:
BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'
!mkdir -p {BACKUP_DIR}/summaries
!cp outputs/summaries/bart_predictions.csv {BACKUP_DIR}/summaries/
print('Predictions backed up to Drive.')

Predictions backed up to Drive.


In [6]:
# 4. Run ROUGE + BERTScore evaluation against the real Summary column
%cd /content/Review-Summarization-LLM
!python -m evaluation.evaluate reference \
    --predictions outputs/summaries/bart_predictions.csv \
    --pred-col prediction --ref-col Summary \
    --output outputs/metrics/bart_results.json

/content/Review-Summarization-LLM
Evaluating 3000 prediction/reference pairs...
config.json: 100% 482/482 [00:00<00:00, 2.50MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 138kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 5.92MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 6.21MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 37.8MB/s]

model.safetensors: downloading bytes:  12% 176M/1.42G [00:01<00:04, 258MB/s, 13.2MB/s  ]
model.safetensors: reconstructing file:   5% 67.0M/1.42G [00:01<00:26, 51.8MB/s]
model.safetensors: downloading bytes:  31% 447M/1.42G [00:01<00:02, 430MB/s, 35.7MB/s  ]
model.safetensors: downloading bytes:  36% 516M/1.42G [00:02<00:02, 394MB/s, 44.2MB/s  ]
model.safetensors: downloading bytes:  50% 712M/1.42G [00:02<00:01, 461MB/s, 59.2MB/s  ]
model.safetensors: downloading bytes:  56% 801M/1.42G [00:02<00:01, 454MB/s, 66.8MB/s  ]
model.safetensors: downloading bytes:  62% 887M/1.42G [00:02<00:01, 445MB/s, 73.9MB/s  ]
model.safetensors: downloading byt

In [7]:
BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'
!mkdir -p {BACKUP_DIR}/metrics
!cp outputs/metrics/bart_results.json {BACKUP_DIR}/metrics/
print('Metrics backed up to Drive.')

Metrics backed up to Drive.


## Next steps

Update `docs/evaluation_report.md` with these ROUGE/BERTScore numbers, and
note the observation (from `01_train_bart.ipynb`'s sanity check) that the
`Summary` column itself is a noisy target -- many reference summaries are
terse or stylistically idiosyncratic review titles rather than descriptive
summaries, which likely caps how low validation/test loss and ROUGE overlap
can realistically go.